# Experiment 3: Overlap Size

## Research Question
How much sliding window overlap is required for stable predictions?

### Fixed Parameters
- Model: gpt-5.4-nano
- Prompt: maximum prompt -> best from experiment 1 -> (minimal prompt for gpt-4o-mini)
- Chunk size: 120 -> best from experiment 2 -> for gpt_5.4-nano is 120 (120 for gpt-4o-mini)
- Few-shot examples: 0

### Variable Parameter
- overlap ∈ {0,10,20}

In [1]:
import sys
project_src = r"C:\Users\sophi\Desktop\Master\CIS_5.Semester\Thesis\locisimiles\src"

# lokalen Pfad *vor* site-packages setzen
sys.path.insert(0, project_src)

import locisimiles
print("Jetzt geladen aus:", locisimiles.__file__)

#imports
import openai
from unittest.mock import patch
import locisimiles as ls
from locisimiles.evaluator import IntertextEvaluator
#from locisimiles.llm_pipeline_new import LLMOnlyClassificationPipeline
from locisimiles.llm_pipeline_sliding_windows import LLMOnlySlidingWindowPipeline, pretty_print, save_final_results, load_final_results
from locisimiles.document import Document, TextSegment
#from locisimiles.llm_pipeline_new import pretty_print
from locisimiles.llm_pipeline_sliding_windows import convert_llm_results, fixed_sliding_windows, convert_llm_results_coverage_fair, pad_segments
import pandas as pd
import csv
from pathlib import Path
from locisimiles.evaluator import IntertextEvaluator
import random
import tempfile
import time

# load key
from dotenv import load_dotenv
load_dotenv()

Jetzt geladen aus: C:\Users\sophi\Desktop\Master\CIS_5.Semester\Thesis\locisimiles\src\locisimiles\__init__.py


c:\Users\sophi\anaconda3\envs\py311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# =========================
# DATA PATH
# =========================
DATA_ROOT = Path("../Sampled_Documents/folds")

N_FOLDS = 5

In [3]:
def load_fold(
    fold_id: int,
    *,
    root: Path = DATA_ROOT,
):
    """
    Load query document, source document and ground truth for one fold.
    """

    fold_dir = root / f"fold_{fold_id}" / "evaluation"

    if not fold_dir.exists():
        raise FileNotFoundError(f"Fold {fold_id} not found at {fold_dir}")

    query_doc = Document(fold_dir / "query_document.csv")
    source_doc = Document(fold_dir / "source_document.csv")
    ground_truth_path = fold_dir / "ground_truth.csv"

    return {
        "query": query_doc,
        "source": source_doc,
        "ground_truth": ground_truth_path,
    }

In [4]:
# =========================
# FIXED PARAMETERS
# =========================

MODEL = "gpt-5.4-nano"
BEST_PROMPT = "zero_shot_maximum" #replace with result from Experiment 1 -> maximum for gpt_5.4-mini(result zero shot minimal for gpt-4o-mini)
BEST_CHUNK = 120 #replace with result from Experiment 2-> 120 for gpt-5.4-mini (result chunk_size 120 for gpt-4o-mini)

NUM_FEW_SHOT = 0

# =========================
# VARIABLE/EXPERIMENT PARAMETER
# =========================
OVERLAPS = [0,10,20]
#OVERLAPS = [20]

# ==============================
# OUTPUT
# ==============================

RESULT_DIR = Path("final_results/experiment_3_overlap")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# =========================
# RUN EXPERIMENT
# =========================
for overlap in OVERLAPS:

    print("\n====================================")
    print(f"Running experiment for overlap: {overlap}")
    print("====================================")

    pipeline = LLMOnlySlidingWindowPipeline(
        openai_model=MODEL,
        prompt_type=BEST_PROMPT,
        chunk_size=BEST_CHUNK,
        overlap=overlap,
        num_few_shot_examples=NUM_FEW_SHOT,
    )

    for fold_id in range(N_FOLDS):

        print(f"\nRunning fold {fold_id}")

        fold = load_fold(fold_id)

        checkpoint_path = RESULT_DIR / f"checkpoint_fold_{fold_id}_overlap{overlap}.json"
        
        start_time = time.perf_counter()

        results = pipeline.run(
            query=fold["query"],
            source=fold["source"],
            checkpoint_path=checkpoint_path,
            resume=True,
        )
        
        end_time = time.perf_counter()
        runtime = end_time - start_time
        print(f"Fold {fold_id} runtime: {runtime:.2f} seconds")

        out_path = RESULT_DIR / f"fold_{fold_id}_overlap_{overlap}.json"

        save_final_results(
            results,
            path=out_path,
            meta={
                "model": pipeline.openai_model,
                "prompt_type": BEST_PROMPT,
                "chunk_size": pipeline.chunk_size,
                "num_few_shot_examples": pipeline.num_few_shot_examples,
                "overlap": pipeline.overlap,
                "runtime_seconds": runtime,
            },
        )

        print(f"Saved results to {out_path}")


Running experiment for overlap: 0

Running fold 0
Total source windows: 9


Processing source windows:   0%|          | 0/9 [00:00<?, ?it/s]

Processing source windows: 100%|██████████| 9/9 [10:00<00:00, 66.72s/it]


Fold 0 runtime: 600.46 seconds
Saved results to final_results\experiment_3_overlap\fold_0_overlap_0.json

Running fold 1
Total source windows: 9


Processing source windows: 100%|██████████| 9/9 [09:43<00:00, 64.78s/it]


Fold 1 runtime: 583.12 seconds
Saved results to final_results\experiment_3_overlap\fold_1_overlap_0.json

Running fold 2
Total source windows: 9


... (22 outputs left out)

Processing source windows: 100%|██████████| 10/10 [13:24<00:00, 80.42s/it]


Fold 3 runtime: 804.21 seconds
Saved results to final_results\experiment_3_overlap\fold_3_overlap_20.json

Running fold 4
Total source windows: 10


Processing source windows: 100%|██████████| 10/10 [13:20<00:00, 80.07s/it]


Fold 4 runtime: 800.75 seconds
Saved results to final_results\experiment_3_overlap\fold_4_overlap_20.json


In [6]:
# =========================
# EVALUATION
# =========================
all_scores = []

for overlap in OVERLAPS:

    print("\n===============================")
    print(f"Evaluating overlap: {overlap}")
    print("===============================")

    for fold_id in range(N_FOLDS):

        print(f"\nEvaluating fold {fold_id}")

        fold = load_fold(fold_id)

        result_file = RESULT_DIR / f"fold_{fold_id}_overlap_{overlap}.json"

        raw_results = load_final_results(result_file)

        pipeline = LLMOnlySlidingWindowPipeline(
            openai_model=MODEL,
            prompt_type=BEST_PROMPT,
            chunk_size=BEST_CHUNK,
            overlap=overlap,
            num_few_shot_examples=NUM_FEW_SHOT,
        )
        
        source_segments = pad_segments(
            list(fold["source"].segments.values()),
            BEST_CHUNK,
            prefix="S"
        )

        query_segments = pad_segments(
            list(fold["query"].segments.values()),
            BEST_CHUNK,
            prefix="Q"
        )

        source_windows = fixed_sliding_windows(
            source_segments,
            BEST_CHUNK,
            pipeline.stride
        )

        query_windows = fixed_sliding_windows(
            query_segments,
            BEST_CHUNK,
            pipeline.stride
        )

        full_results = convert_llm_results_coverage_fair(
            raw_results,
            fold["source"],
            fold["query"],
            source_windows,
            query_windows,
            threshold=0.0
        )

        wrapped_pipeline = type(
            "DummyPipeline",
            (),
            {"run": lambda self, **kwargs: full_results}
        )()

        evaluator = IntertextEvaluator(
            query_doc=fold["query"],
            source_doc=fold["source"],
            ground_truth_csv=fold["ground_truth"],
            pipeline=wrapped_pipeline,
            top_k=1000,
            threshold=0.5,
        )

        macro_scores = evaluator.evaluate(average="macro")
        micro_scores = evaluator.evaluate(average="micro")

        print("Macro:", macro_scores)
        print("Micro:", micro_scores)
        
        macro_row = macro_scores.iloc[0]
        micro_row = micro_scores.iloc[0]

        all_scores.append({
            "overlap": overlap,
            "fold": fold_id,
            "macro_precision": macro_row["precision"],
            "macro_recall": macro_row["recall"],
            "macro_f1": macro_row["f1"],
            "macro_accuracy": macro_row["accuracy"],
            "macro_fpr": macro_row["fpr"],
            "macro_fnr": macro_row["fnr"],
            "macro_smr": macro_row["smr"],
            "macro_tp": macro_row["tp"],
            "macro_fp": macro_row["fp"],
            "macro_fn": macro_row["fn"],
            "macro_tn": macro_row["tn"],

            "micro_precision": micro_row["precision"],
            "micro_recall": micro_row["recall"],
            "micro_f1": micro_row["f1"],
            "micro_accuracy": micro_row["accuracy"],
            "micro_fpr": micro_row["fpr"],
            "micro_fnr": micro_row["fnr"],
            "micro_smr": micro_row["smr"],
            "micro_tp": micro_row["tp"],
            "micro_fp": micro_row["fp"],
            "micro_fn": micro_row["fn"],
            "micro_tn": micro_row["tn"],
        })


Evaluating overlap: 0

Evaluating fold 0
Macro:    precision    recall        f1  accuracy       fpr      fnr       smr  tp  \
0   0.018143  0.018677  0.018143  0.999885  0.000005  0.00011  0.000115  18   

   fp  fn      tn  
0   4  91  824447  
Micro:    precision    recall        f1  accuracy       fpr      fnr       smr  tp  \
0   0.818182  0.165138  0.274809  0.999885  0.000005  0.00011  0.000115  18   

   fp  fn      tn  
0   4  91  824447  

Evaluating fold 1
Macro:    precision    recall       f1  accuracy       fpr       fnr       smr  tp  \
0   0.029947  0.032086  0.03066  0.999891  0.000013  0.000096  0.000109  30   

   fp  fn      tn  
0  11  79  822680  
Micro:    precision    recall   f1  accuracy       fpr       fnr       smr  tp  fp  \
0   0.731707  0.275229  0.4  0.999891  0.000013  0.000096  0.000109  30  11   

   fn      tn  
0  79  822680  

Evaluating fold 2
Macro:    precision    recall        f1  accuracy       fpr       fnr       smr  tp  \
0   0.021345  0.0

In [7]:
# =========================
# RESULTS AGGREGATION
# =========================
df = pd.DataFrame(all_scores)

summary = (
    df.groupby("overlap")
    .mean(numeric_only=True)
    .sort_values("macro_f1", ascending=False)
)

summary

,fold,macro_precision,macro_recall,macro_f1,macro_accuracy,macro_fpr,macro_fnr,macro_smr,macro_tp,macro_fp,...,micro_recall,micro_f1,micro_accuracy,micro_fpr,micro_fnr,micro_smr,micro_tp,micro_fp,micro_fn,micro_tn
overlap,,,,,,,,,,,,,,,,,,,,,
20,2.0,0.042168,0.045507,0.043108,0.999904,0.000017,0.000079,0.000096,43.4,14.0,...,0.398879,0.521138,0.999904,0.000017,0.000079,0.000096,43.4,14.0,65.4,823029.2
10,2.0,0.031338,0.032925,0.031780,0.999897,0.000010,0.000094,0.000103,31.6,8.0,...,0.290418,0.424565,0.999897,0.000010,0.000094,0.000103,31.6,8.0,77.2,823035.2
0,2.0,0.031306,0.032729,0.031697,0.999898,0.000008,0.000094,0.000102,31.0,6.4,...,0.285134,0.415588,0.999898,0.000008,0.000094,0.000102,31.0,6.4,77.8,823036.8


In [10]:
df

,overlap,fold,macro_precision,macro_recall,macro_f1,macro_accuracy,macro_fpr,macro_fnr,macro_smr,macro_tp,...,micro_recall,micro_f1,micro_accuracy,micro_fpr,micro_fnr,micro_smr,micro_tp,micro_fp,micro_fn,micro_tn
0,0,0,0.018143,0.018677,0.018143,0.999885,0.000005,0.000110,0.000115,18.0,...,0.165138,0.274809,0.999885,0.000005,0.000110,0.000115,18.0,4.0,91.0,824447.0
1,0,1,0.029947,0.032086,0.030660,0.999891,0.000013,0.000096,0.000109,30.0,...,0.275229,0.400000,0.999891,0.000013,0.000096,0.000109,30.0,11.0,79.0,822680.0
2,0,2,0.021345,0.021878,0.021345,0.999888,0.000005,0.000107,0.000112,21.0,...,0.192661,0.313433,0.999888,0.000005,0.000107,0.000112,21.0,4.0,88.0,824447.0
3,0,3,0.045161,0.045161,0.045161,0.999918,0.000001,0.000081,0.000082,43.0,...,0.394495,0.562092,0.999918,0.000001,0.000081,0.000082,43.0,1.0,66.0,818290.0
4,0,4,0.041933,0.045842,0.043177,0.999907,0.000015,0.000079,0.000093,43.0,...,0.398148,0.527607,0.999907,0.000015,0.000079,0.000093,43.0,12.0,65.0,825320.0
5,10,0,0.036286,0.037887,0.036820,0.999904,0.000008,0.000087,0.000096,37.0,...,0.339450,0.483660,0.999904,0.000008,0.000087,0.000096,37.0,7.0,72.0,824444.0
6,10,1,0.021747,0.024064,0.022531,0.999883,0.000012,0.000105,0.000117,23.0,...,0.211009,0.323944,0.999883,0.000012,0.000105,0.000117,23.0,10.0,86.0,822681.0
7,10,2,0.035646,0.036820,0.035829,0.999899,0.000012,0.000089,0.000101,36.0,...,0.330275,0.464516,0.999899,0.000012,0.000089,0.000101,36.0,10.0,73.0,824441.0
8,10,3,0.033871,0.033871,0.033692,0.999905,0.000001,0.000094,0.000095,32.0,...,0.293578,0.450704,0.999905,0.000001,0.000094,0.000095,32.0,1.0,77.0,818290.0
9,10,4,0.029140,0.031983,0.030028,0.999891,0.000015,0.000094,0.000109,30.0,...,0.277778,0.400000,0.999891,0.000015,0.000094,0.000109,30.0,12.0,78.0,825320.0


In [8]:
# =========================
# SAVE RESULT EVALUATION
# =========================
summary_path = RESULT_DIR / "overlap_experiment_summary.csv"

summary.to_csv(summary_path)
df.to_csv(RESULT_DIR / "overlap_experiment_results.csv", index=False)

print("Saved summary to", summary_path)

Saved summary to final_results\experiment_3_overlap\overlap_experiment_summary.csv
